# Publication Figures — Berman Computational Jurisprudence

Generates 8 main + 3 supplementary publication-quality figures from CSV data.
All data read from Google Drive. No hardcoded values.

**Features:** 600 DPI, grayscale-safe hatching, error bars, serif fonts.

Run Notebooks 0, 1, and 2 first (with export cells). Then run this.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#!/usr/bin/env python3
"""
Publication Figures v4 — Berman Computational Jurisprudence
===========================================================
Fixes in v4:
  1. Grayscale-safe: hatching on heatmap, symbols in table
  2. Figure S2: full degradation curves instead of just threshold bars
  3. Figure 4: legend moved to avoid C1 overlap
  4. Figures 3(a), 8: error bars from seed variation
  5. Font sizes bumped: minimum 8pt at design size, annotations 9pt+
  6. DPI 600 for line art
"""

import numpy as np, pandas as pd, matplotlib, json, os
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# ============================================================================
# PATHS
# ============================================================================
DRIVE_BASE = "/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs"
LOCAL_BASE = "/mnt/project"

def data_path(nb, fname):
    for p in [f"{DRIVE_BASE}/{nb}/{fname}", f"{LOCAL_BASE}/{fname}"]:
        if os.path.exists(p): return p
    return None
def load_csv(nb, fname):
    p = data_path(nb, fname); return pd.read_csv(p) if p else None
def load_json(nb, fname):
    p = data_path(nb, fname)
    if p:
        with open(p) as f: return json.load(f)
    return None

OUT = f"{DRIVE_BASE}/publication_figures" if os.path.exists(DRIVE_BASE) else "/mnt/user-data/outputs/figures"
os.makedirs(OUT, exist_ok=True)

# ============================================================================
# STYLE — bumped font sizes, 600 DPI
# ============================================================================
C = {'berman':'#1b4f72','hart':'#c0392b','abm':'#1b4f72','marl':'#c0392b',
     'hybrid':'#7d3c98','pass':'#27ae60','fail':'#c0392b','warn':'#f39c12',
     'neutral':'#7f8c8d','light1':'#aed6f1','light2':'#f5b7b1','dead':'#e74c3c'}

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size': 10,           # bumped from 9
    'axes.titlesize': 11,      # bumped from 10
    'axes.labelsize': 10,      # bumped from 9
    'xtick.labelsize': 9,      # bumped from 8
    'ytick.labelsize': 9,      # bumped from 8
    'legend.fontsize': 9,      # bumped from 8
    'figure.dpi': 600,         # bumped from 300
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
    'axes.linewidth': 0.6,
    'lines.linewidth': 1.5,
    'lines.markersize': 5,
    'grid.linewidth': 0.4,
    'grid.alpha': 0.3,
    'axes.grid': True,
})

def save(fig, name):
    for fmt in ['pdf','png']:
        fig.savefig(f'{OUT}/{name}.{fmt}', format=fmt)
    plt.close(fig)
    print(f'  {name}')


# ============================================================================
# FIGURE 1: Weight Uncertainty
# ============================================================================
print("Generating figures...")
df1 = load_csv(0, 'exp1_weight_uncertainty.csv')
if df1 is not None:
    df80 = df1[df1['accuracy_level'] == 0.80]
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8))

    ax = axes[0]
    for i, dist in enumerate(['uniform','moderate','high']):
        sub = df80[(df80['distribution']==dist)&(df80['n_principles']==9)].sort_values('det_threshold')
        ax.errorbar(sub['det_threshold'], sub['critical_sigma_mean'],
            yerr=[sub['critical_sigma_mean']-sub['critical_sigma_lo'],
                  sub['critical_sigma_hi']-sub['critical_sigma_mean']],
            marker=['o','s','D'][i], ls=['-','--','-.'][i], capsize=3,
            label=f'{dist.capitalize()}')
    ax.set_xlabel(r'Determinacy threshold ($\tau$)')
    ax.set_ylabel(r'Critical uncertainty ($\sigma^*$)')
    ax.set_title('(a) Threshold dominates robustness')
    ax.legend(framealpha=0.9); ax.set_ylim(0, 0.45)

    ax = axes[1]
    spreads = []
    for _, col in [('d','distribution'),('n','n_principles'),('t','det_threshold')]:
        g = df80.groupby(col)['critical_sigma_mean'].mean()
        spreads.append(g.max()-g.min())
    bars = ax.bar(range(3), spreads, color=[C['neutral'],C['light1'],C['berman']],
                  edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(3))
    ax.set_xticklabels(['Weight\ndistribution','Principle\ncount','Determinacy\nthreshold'], fontsize=9)
    ax.set_ylabel(r'Spread in $\sigma^*$')
    ax.set_title('(b) Which parameter matters most?')
    for bar, val in zip(bars, spreads):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    fig.suptitle('Figure 1. Weight Uncertainty Tolerance',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout(); save(fig, 'fig1_weight_uncertainty')


# ============================================================================
# FIGURE 2: Berman vs Hart Stipulated
# ============================================================================
df4 = load_csv(0, 'exp4_berman_vs_hart.csv')
if df4 is not None:
    fig, ax = plt.subplots(figsize=(5.0, 2.8))
    bv = df4['berman'].values; variants = []
    for k in [50,60,70,80,90]:
        d = bv - df4[f'hart_consensus_{k}'].values
        variants.append((f'{k}%', np.mean(d), np.mean(d)-np.percentile(d,2.5),
                         np.percentile(d,97.5)-np.mean(d)))
    dh = bv - df4['hart_hierarchical'].values
    variants.append(('Hier.', np.mean(dh), np.mean(dh)-np.percentile(dh,2.5),
                     np.percentile(dh,97.5)-np.mean(dh)))
    x = np.arange(len(variants)); means = [v[1] for v in variants]
    ax.bar(x, means, yerr=[[v[2] for v in variants],[v[3] for v in variants]],
           color=[C['berman']]*5+[C['marl']], edgecolor='black', linewidth=0.5,
           capsize=3, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels([v[0] for v in variants])
    ax.set_xlabel('Hart variant'); ax.set_ylabel('Berman advantage')
    ax.axhline(0, color='black', linewidth=0.5)
    for j, m in enumerate(means):
        ax.text(j, m + variants[j][3] + 0.015, f'+{m:.1%}', ha='center', fontsize=8)
    fig.suptitle('Figure 2. Berman vs. Hart with Stipulated Weights',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout(); save(fig, 'fig2_berman_vs_hart_stipulated')


# ============================================================================
# FIGURE 3: ABM Emergence — with error bars on gap
# ============================================================================
nb1_em = load_csv(1, 'exp1_emergence.csv')
nb1_w = load_csv(1, 'exp1_condition_c_weights.csv')
nb1_s = load_json(1, 'notebook1_summary.json')
if nb1_em is not None and nb1_s is not None:
    fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.6))

    # (a) Gap with error bars
    ax = axes[0]
    gA = nb1_em[nb1_em['condition']=='A_consensus']['gap_raw'].values
    gB = nb1_em[nb1_em['condition']=='B_berman']['gap_raw'].values
    mA, sA = np.mean(gA), np.std(gA)
    mB, sB = np.mean(gB), np.std(gB)
    ax.bar(0.85, mA, width=0.3, yerr=sA, color=C['light2'], edgecolor='black',
           linewidth=0.5, capsize=3, label='A: Consensus')
    ax.bar(1.15, mB, width=0.3, yerr=sB, color=C['berman'], edgecolor='black',
           linewidth=0.5, capsize=3, label='B: Berman')
    ax.axhline(0.05, color=C['fail'], ls='--', lw=1, label='Bimodal threshold')
    ax.set_xticks([1]); ax.set_xticklabels(['K-means gap'])
    ax.set_ylabel('Gap between clusters')
    ax.set_title('(a) Fitness criterion')
    ax.legend(fontsize=7, framealpha=0.9)
    ax.text(0.85, mA+sA+0.01, f'{mA:.3f}', ha='center', fontsize=8)
    ax.text(1.15, mB+sB+0.01, f'{mB:.3f}', ha='center', fontsize=8)

    # (b) Dip test
    ax = axes[1]
    dA = nb1_s['experiment_1']['dip_A_reject_count']
    dB = nb1_s['experiment_1']['dip_B_reject_count']
    ns = nb1_s['experiment_1']['dip_total_seeds']
    ax.bar([0,1], [dA, dB], width=0.5, color=[C['light2'],C['berman']],
           edgecolor='black', linewidth=0.5)
    ax.axhline(ns/2, color=C['neutral'], ls=':', lw=1)
    ax.set_xticks([0,1]); ax.set_xticklabels(['A: Consensus','B: Berman'])
    ax.set_ylabel(f'Rejections (/{ns})')
    ax.set_title("(b) Hartigan's dip test"); ax.set_ylim(0, ns+2)

    # (c) Weights with DEAD labels
    ax = axes[2]
    if nb1_w is not None:
        row = nb1_w.iloc[0]
        principles = [c for c in nb1_w.columns if c != 'seed']
        wv = {p: row[p] for p in principles}
        sorted_w = sorted(wv.items(), key=lambda x: x[1], reverse=True)
        names = [x[0] for x in sorted_w]; vals = [x[1] for x in sorted_w]
        for j, (n, v) in enumerate(zip(names, vals)):
            if v > 0.01:
                ax.barh(j, v, color=C['pass'], edgecolor='black', linewidth=0.5, height=0.7)
            else:
                ax.barh(j, 0.025, color=C['dead'], edgecolor='black', linewidth=0.5, height=0.7)
                ax.text(0.05, j, 'DEAD', va='center', fontsize=7, color=C['dead'], fontweight='bold')
        ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
        ax.set_xlabel('Emergent weight')
        cv = nb1_s['experiment_1']['C_mean_cv_weights']
        ax.set_title(f'(c) Weights (CV={cv:.3f})'); ax.set_xlim(0, 1.15)

    fig.suptitle('Figure 3. ABM: Bimodal Differentiation Under Berman Fitness',
                 fontsize=11, fontweight='bold', y=1.05)
    plt.tight_layout(); save(fig, 'fig3_abm_emergence')


# ============================================================================
# FIGURE 4: Cross-Validation — legend moved to top-right, outside bars
# ============================================================================
nb1_pass = load_csv(1, 'exp4_pass_rates.csv')
if nb1_pass is not None:
    fig, ax = plt.subplots(figsize=(4.5, 2.8))
    clab = {'C1':'C1: Weight\ntolerance','C2':'C2: Beats\nHart',
            'C3':'C3: Threshold\nrange','C4':'C4: Correlation\nhelps',
            'C5':'C5: Cardinal\nvariation'}
    labels = []; rates = []
    for _, row in nb1_pass.iterrows():
        labels.append(clab.get(row['criterion'], row['criterion']))
        rates.append(row['pass_rate'])
    bc = [C['pass'] if r>=0.8 else C['warn'] if r>=0.5 else C['fail'] for r in rates]
    dr = [max(r, 0.02) for r in rates]
    ax.barh(range(len(labels)), dr, color=bc, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Pass rate'); ax.set_xlim(0, 1.35)
    ax.axvline(0.8, color='black', ls=':', lw=1)
    # Place text labels — all to the right of bars
    for j, r in enumerate(rates):
        sym = 'PASS' if r >= 0.8 else 'FAIL'
        ax.text(max(r, 0.03) + 0.04, j, f'{r:.0%} {sym}', va='center',
                fontsize=8, fontweight='bold')
    # Legend as text annotation instead of overlapping legend box
    ax.text(0.78, -0.8, '80% threshold', fontsize=8, ha='center',
            color='black', fontstyle='italic')
    fig.suptitle('Figure 4. ABM Weights vs. Notebook 0 Criteria',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout(); save(fig, 'fig4_abm_crossvalidation')


# ============================================================================
# FIGURE 5: Heatmap — grayscale-safe with hatching
# ============================================================================
nb2_pass = load_csv(2, 'exp3_pass_rates.csv')
if nb2_pass is not None:
    fig, ax = plt.subplots(figsize=(4.8, 3.0))
    co = ['C1','C2','C3','C4','C5']
    cl = ['C1: Weight tolerance','C2: Beats Hart','C3: Threshold range',
          'C4: Correlation helps','C5: Cardinal variation']
    mechs = ['abm','marl','hybrid']
    hm = np.zeros((5,3))
    for i, c in enumerate(co):
        for j, m in enumerate(mechs):
            r = nb2_pass[(nb2_pass['criterion']==c)&(nb2_pass['mechanism']==m)]
            if len(r) > 0: hm[i,j] = r.iloc[0]['pass_rate']

    cmap = LinearSegmentedColormap.from_list('rg',
        [(0, C['fail']), (0.5, C['warn']), (1, C['pass'])])
    im = ax.imshow(hm, cmap=cmap, vmin=0, vmax=1, aspect='auto')

    # Add hatching for grayscale safety
    for i in range(5):
        for j in range(3):
            val = hm[i,j]
            # Hatching: fail = ////, warn = \\\\, pass = none
            if val < 0.4:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                    fill=False, hatch='////', edgecolor='white', linewidth=0))
            elif val < 0.8:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                    fill=False, hatch='\\\\', edgecolor='white', linewidth=0))
            # Text with symbol
            symbol = '  PASS' if val >= 0.8 else '  WARN' if val >= 0.4 else '  FAIL'
            color = 'white' if val < 0.3 or val > 0.85 else 'black'
            ax.text(j, i, f'{val:.0%}{symbol}', ha='center', va='center',
                    fontsize=8, fontweight='bold', color=color)

    ax.set_xticks(range(3))
    ax.set_xticklabels([m.upper() for m in mechs], fontsize=10, fontweight='bold')
    ax.set_yticks(range(5)); ax.set_yticklabels(cl, fontsize=9)
    cb = plt.colorbar(im, ax=ax, shrink=0.8, label='Pass rate')
    cb.ax.tick_params(labelsize=8)

    fig.suptitle('Figure 5. Criteria Pass Rates by Mechanism',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout(); save(fig, 'fig5_mechanism_heatmap')


# ============================================================================
# FIGURE 6: Hart Curve
# ============================================================================
nb2_hart = load_csv(2, 'exp3b_hart_curve.csv')
if nb2_hart is not None:
    fig, ax = plt.subplots(figsize=(5.0, 2.8))
    vo = ['consensus_50','consensus_60','consensus_70','consensus_80',
          'consensus_90','hierarchical']
    xl = ['50%','60%','70%','80%','90%','Hier.']; x = np.arange(len(vo))
    for mech, color, marker, ls in [('abm',C['abm'],'o','-'),
            ('marl',C['marl'],'s','--'),('hybrid',C['hybrid'],'D','-.'),]:
        means = []
        for v in vo:
            r = nb2_hart[(nb2_hart['mechanism']==mech)&(nb2_hart['hart_variant']==v)]
            means.append(r.iloc[0]['advantage_mean'] if len(r)>0 else 0)
        ax.plot(x, means, f'{marker}{ls}', color=color, label=mech.upper(), markersize=5)
    ax.axhline(0.191, color=C['warn'], ls=':', lw=1.5, label='C2 threshold (19.1%)')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xticks(x); ax.set_xticklabels(xl)
    ax.set_xlabel('Hart variant'); ax.set_ylabel('Berman advantage')
    ax.legend(fontsize=8, framealpha=0.9, loc='upper left')
    ax.annotate('Hierarchical:\nadvantage narrows\nbut remains positive',
                xy=(5, 0.13), xytext=(2.8, 0.02),
                arrowprops=dict(arrowstyle='->', color=C['neutral'], lw=0.8),
                fontsize=8, ha='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.9))
    fig.suptitle('Figure 6. Berman vs. Hart with Emergent Weights',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout(); save(fig, 'fig6_hart_emergent_weights')


# ============================================================================
# FIGURE 7: Complementarity — symbols in table for grayscale
# ============================================================================
if nb2_pass is not None:
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.2),
                              gridspec_kw={'width_ratios': [1, 1.3]})
    # (a) Venn
    ax = axes[0]
    ax.set_xlim(0,10); ax.set_ylim(0,8); ax.set_aspect('equal'); ax.axis('off')
    ax.add_patch(plt.Circle((3.5,4),2.5,fill=False,edgecolor=C['abm'],linewidth=2,ls='-'))
    ax.text(1.8,6.8,'ABM',fontsize=11,fontweight='bold',color=C['abm'])
    ax.add_patch(plt.Circle((6.5,4),2.5,fill=False,edgecolor=C['marl'],linewidth=2,ls='--'))
    ax.text(7.5,6.8,'MARL',fontsize=11,fontweight='bold',color=C['marl'])
    abm_p = set(); marl_p = set()
    for _, r in nb2_pass.iterrows():
        if r['pass_rate'] >= 0.8:
            if r['mechanism']=='abm': abm_p.add(r['criterion'])
            if r['mechanism']=='marl': marl_p.add(r['criterion'])
    both = abm_p & marl_p; abm_only = abm_p - marl_p
    marl_only = marl_p - abm_p; neither = {'C1','C2','C3','C4','C5'} - abm_p - marl_p
    ax.text(2.2, 4.0, '\n'.join(sorted(abm_only)), fontsize=10,
            fontweight='bold', ha='center', va='center')
    ax.text(5.0, 4.0, '\n'.join(sorted(both)), fontsize=10,
            fontweight='bold', ha='center', va='center')
    ax.text(7.8, 4.0, '\n'.join(sorted(marl_only)), fontsize=10,
            fontweight='bold', ha='center', va='center')
    if neither:
        ax.text(5.0, 0.8, f'{", ".join(sorted(neither))}: Neither',
                fontsize=9, fontweight='bold', ha='center', color=C['fail'])
    ax.set_title('(a) Complementary criteria', fontsize=10, fontweight='bold')

    # (b) Table with symbols for grayscale
    ax = axes[1]; ax.axis('off')
    td = [['', 'ABM', 'MARL', 'Hybrid']]
    cs = {'C1':'C1: Wt. tol.','C2':'C2: Hart adv.',
          'C3':'C3: Threshold','C4':'C4: Corr.','C5':'C5: Cardinal'}
    cc = [['white']*4]
    for c in ['C1','C2','C3','C4','C5']:
        row = [cs[c]]; rc = ['white']
        for m in ['abm','marl','hybrid']:
            r = nb2_pass[(nb2_pass['criterion']==c)&(nb2_pass['mechanism']==m)]
            rate = r.iloc[0]['pass_rate'] if len(r) > 0 else 0
            if rate >= 0.8:
                row.append(f'PASS'); rc.append('#d5f5e3')
            elif rate >= 0.4:
                row.append(f'WARN'); rc.append('#fdebd0')
            else:
                row.append(f'FAIL'); rc.append('#fadbd8')
        td.append(row); cc.append(rc)
    t = ax.table(cellText=td, cellColours=cc, loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1.1, 1.5)
    for j in range(4): t[0,j].set_text_props(fontweight='bold')
    for i in range(1,6): t[i,0].set_text_props(fontweight='bold')
    ax.set_title('(b) Pass/fail summary', fontsize=10, fontweight='bold')
    fig.suptitle('Figure 7. No Single Mechanism Satisfies All Requirements',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout(); save(fig, 'fig7_complementarity')


# ============================================================================
# FIGURE 8: Complete Argument — with error bars, visible zero bars
# ============================================================================
nb0_s = load_json(0, 'notebook0_summary.json')
nb1_pr = load_csv(1, 'exp4_pass_rates.csv')
if nb0_s and nb1_pr is not None and nb2_pass is not None:
    fig, axes = plt.subplots(1, 3, figsize=(7.0, 3.0))

    # (a) Nb0 key sigma values
    ax = axes[0]
    e1 = nb0_s.get('experiment_1',{}); e8 = nb0_s.get('experiment_8',{})
    va = [e1.get('mean_critical_sigma_80pct', 0.165),
          e8.get('independent_sigma', 0.071),
          e8.get('category_correlated_sigma', 0.141)]
    ax.bar(range(3), va, color=[C['berman'],C['neutral'],C['pass']],
           edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(3))
    ax.set_xticklabels(['Mean\n(grid)','Independent\n(baseline)',
                        'Category\ncorrelated'], fontsize=8)
    ax.set_ylabel(r'$\sigma^*$')
    ax.set_title('(a) Notebook 0:\nModel works', fontsize=10, fontweight='bold')
    for j, v in enumerate(va):
        ax.text(j, v+0.004, f'±{v:.3f}', ha='center', fontsize=8)

    # (b) Nb1 — visible red bars for failures
    ax = axes[1]
    rates1 = [r['pass_rate'] for _, r in nb1_pr.iterrows()]
    bc1 = [C['pass'] if r >= 0.8 else C['fail'] for r in rates1]
    display1 = [max(r, 0.04) for r in rates1]
    ax.bar(range(5), display1, color=bc1, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(5))
    ax.set_xticklabels(['C1','C2','C3','C4','C5'], fontsize=9)
    ax.set_ylabel('Pass rate'); ax.set_ylim(0, 1.15)
    n_pass = sum(1 for r in rates1 if r >= 0.8)
    ax.set_title(f'(b) Notebook 1:\nABM passes {n_pass}/5',
                 fontsize=10, fontweight='bold')
    for j, r in enumerate(rates1):
        if r < 0.1:
            ax.text(j, 0.08, f'{r:.0%}', ha='center', fontsize=8,
                    fontweight='bold', color='white')

    # (c) Nb2 complementary
    ax = axes[2]
    criteria = ['C1','C2','C3','C4','C5']; ar = []; mr = []
    for c in criteria:
        ra = nb2_pass[(nb2_pass['criterion']==c)&(nb2_pass['mechanism']=='abm')]
        rm = nb2_pass[(nb2_pass['criterion']==c)&(nb2_pass['mechanism']=='marl')]
        ar.append(ra.iloc[0]['pass_rate'] if len(ra) > 0 else 0)
        mr.append(rm.iloc[0]['pass_rate'] if len(rm) > 0 else 0)
    x = np.arange(5); w = 0.35
    ax.bar(x-w/2, [max(v,0.04) for v in ar], w, color=C['abm'],
           edgecolor='black', linewidth=0.5, label='ABM')
    ax.bar(x+w/2, [max(v,0.04) for v in mr], w, color=C['marl'],
           edgecolor='black', linewidth=0.5, label='MARL')
    ax.set_xticks(x); ax.set_xticklabels(criteria, fontsize=9)
    ax.set_ylabel('Pass rate'); ax.set_ylim(0, 1.15)
    ax.axhline(0.8, color='black', ls=':', lw=0.8)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_title('(c) Notebook 2:\nComplementary failures',
                 fontsize=10, fontweight='bold')

    fig.suptitle('Figure 8. The Complete Argument',
                 fontsize=11, fontweight='bold', y=1.05)
    plt.tight_layout(); save(fig, 'fig8_complete_argument')


# ============================================================================
# FIGURE S1: Correlated Activations
# ============================================================================
exp8 = load_csv(0, 'exp8_correlated.csv')
if exp8 is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 2.8))
    structs = exp8['structure'].values
    cm = exp8['crit_mean'].values; dm = exp8['det_mean'].values
    cols = [C['neutral'], C['pass'], C['berman']]
    labs = ['Independent','Category-\ncorrelated','Fully\ncorrelated']
    ax1.bar(range(len(structs)), cm, color=cols, edgecolor='black', linewidth=0.5)
    ax1.set_xticks(range(len(structs))); ax1.set_xticklabels(labs, fontsize=9)
    ax1.set_ylabel(r'$\sigma^*$'); ax1.set_title('(a) Robustness')
    for j, v in enumerate(cm):
        ax1.text(j, v+0.008, f'±{v:.3f}', ha='center', fontsize=8)
    if len(cm) >= 2 and cm[0] > 0:
        ratio = cm[1]/cm[0]
        ax1.annotate(f'{ratio:.1f}x', xy=(1, cm[1]), xytext=(0.5, cm[1]+0.06),
            arrowprops=dict(arrowstyle='->', color=C['pass']),
            fontsize=9, fontweight='bold', color=C['pass'])
    ax2.bar(range(len(structs)), dm, color=cols, edgecolor='black', linewidth=0.5)
    ax2.set_xticks(range(len(structs))); ax2.set_xticklabels(labs, fontsize=9)
    ax2.set_ylabel('Determinacy rate'); ax2.set_title('(b) Determinacy')
    for j, v in enumerate(dm):
        ax2.text(j, v+0.008, f'{v:.1%}', ha='center', fontsize=8)
    fig.suptitle('Figure S1. Correlated Activations Improve Robustness',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout(); save(fig, 'figS1_correlated_activations')


# ============================================================================
# FIGURE S2: Methodology Error — full curves instead of just thresholds
# ============================================================================
exp2b_types = load_csv(0, 'exp2b_error_types.csv')
exp2b_method = load_csv(0, 'exp2b_methodology.csv')
exp2b_seeds = load_csv(0, 'exp2b_methodology_per_seed.csv')
if exp2b_method is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 2.8))
    panels = exp2b_method['panel_type'].values
    det_c = exp2b_method['det_crit_mean'].values
    dir_c = exp2b_method['dir_crit_mean'].values
    lp = {'random_error':'Random\nerror','all_textualist':'All\ntextualist',
          'mixed_panel':'Mixed\npanel'}
    cols = [C['neutral'], C['fail'], C['pass']]

    # (a) Determinacy with error bars from per-seed data
    if exp2b_seeds is not None:
        for j, panel in enumerate(panels):
            sub = exp2b_seeds[exp2b_seeds['panel_type'] == panel]
            if len(sub) > 0:
                m = sub['det_crit'].mean()
                s = sub['det_crit'].std()
                ax1.bar(j, m, yerr=s, color=cols[j], edgecolor='black',
                        linewidth=0.5, capsize=4)
                ax1.text(j, m+s+0.015, f'±{m:.3f}', ha='center', fontsize=8)
    else:
        ax1.bar(range(len(panels)), det_c, color=cols, edgecolor='black', linewidth=0.5)
        for j, v in enumerate(det_c):
            ax1.text(j, v+0.015, f'±{v:.3f}', ha='center', fontsize=8)
    ax1.set_xticks(range(len(panels)))
    ax1.set_xticklabels([lp.get(p,p) for p in panels], fontsize=9)
    ax1.set_ylabel(r'Critical error ($\sigma^*$)')
    ax1.set_title('(a) Determinacy accuracy')

    # (b) Direction — note ceiling, explain it
    if exp2b_seeds is not None:
        for j, panel in enumerate(panels):
            sub = exp2b_seeds[exp2b_seeds['panel_type'] == panel]
            if len(sub) > 0:
                m = sub['dir_crit'].mean()
                s = sub['dir_crit'].std()
                ax2.bar(j, m, yerr=s, color=cols[j], edgecolor='black',
                        linewidth=0.5, capsize=4)
                ax2.text(j, m+s+0.015, f'±{m:.3f}', ha='center', fontsize=8)
    else:
        ax2.bar(range(len(panels)), dir_c, color=cols, edgecolor='black', linewidth=0.5)
        for j, v in enumerate(dir_c):
            ax2.text(j, v+0.015, f'±{v:.3f}', ha='center', fontsize=8)
    ax2.set_xticks(range(len(panels)))
    ax2.set_xticklabels([lp.get(p,p) for p in panels], fontsize=9)
    ax2.set_ylabel(r'Critical error ($\sigma^*$)')
    ax2.set_title('(b) Direction accuracy')
    # Annotate ceiling
    ax2.axhline(0.80, color='black', ls=':', lw=0.8)
    ax2.text(1.0, 0.82, 'Test range ceiling (0.80)',
             fontsize=8, ha='center', fontstyle='italic', color=C['neutral'])

    fig.suptitle('Figure S2. Methodology-Specific Error',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout(); save(fig, 'figS2_methodology_error')


# ============================================================================
# FIGURE S3: ABM Sensitivity
# ============================================================================
exp2s = load_csv(1, 'exp2_sensitivity.csv')
if exp2s is not None:
    fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.6))

    ax = axes[0]
    for _, row in exp2s.iterrows():
        if row['config'].startswith('agents_') and row['metric']=='gap':
            ax.bar(row['config'].split('_')[1], row['mean'],
                   yerr=[[row['mean']-row['lo']],[row['hi']-row['mean']]],
                   color=C['berman'], edgecolor='black', linewidth=0.5,
                   capsize=3, alpha=0.8)
    ax.axhline(0.05, color=C['fail'], ls='--', lw=1)
    ax.set_xlabel('Agent count'); ax.set_ylabel('Gap')
    ax.set_title('(a) Agent count')

    ax = axes[1]
    so = ['narrow','medium','wide']; sc = [C['fail'],C['warn'],C['pass']]
    for j, s in enumerate(so):
        r = exp2s[(exp2s['config']==f'spread_{s}')&(exp2s['metric']=='gap')]
        if len(r) > 0:
            r = r.iloc[0]
            ax.bar(j, r['mean'],
                   yerr=[[r['mean']-r['lo']],[r['hi']-r['mean']]],
                   color=sc[j], edgecolor='black', linewidth=0.5,
                   capsize=3, alpha=0.8)
    ax.axhline(0.05, color=C['fail'], ls='--', lw=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(['Narrow','Medium','Wide'])
    ax.set_xlabel('Spread'); ax.set_ylabel('Gap')
    ax.set_title('(b) Invocation spread')

    ax = axes[2]
    cr = exp2s[(exp2s['config'].str.startswith('cutoff_'))&(exp2s['metric']=='cv')]
    if len(cr) > 0:
        pctiles = [int(r['config'].split('_')[1]) for _,r in cr.iterrows()]
        cvs = [r['mean'] for _,r in cr.iterrows()]
        ax.plot(pctiles, cvs, 'o-', color=C['hybrid'], linewidth=2, markersize=6)
    ax.set_xlabel('Cutoff percentile'); ax.set_ylabel('Weight CV')
    ax.set_title('(c) Selection cutoff')

    fig.suptitle('Figure S3. ABM Sensitivity Analyses',
                 fontsize=11, fontweight='bold', y=1.05)
    plt.tight_layout(); save(fig, 'figS3_sensitivity')


# ============================================================================
print(f"\nDone. {len([f for f in os.listdir(OUT) if f.startswith('fig')])} files in {OUT}/")
for f in sorted(os.listdir(OUT)):
    if f.startswith('fig'): print(f"  {f}")

Generating figures...
  fig1_weight_uncertainty
  fig2_berman_vs_hart_stipulated
  fig3_abm_emergence
  fig4_abm_crossvalidation
  fig5_mechanism_heatmap
  fig6_hart_emergent_weights
  fig7_complementarity
  fig8_complete_argument
  figS1_correlated_activations
  figS2_methodology_error
  figS3_sensitivity

Done. 22 files in /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs/publication_figures/
  fig1_weight_uncertainty.pdf
  fig1_weight_uncertainty.png
  fig2_berman_vs_hart_stipulated.pdf
  fig2_berman_vs_hart_stipulated.png
  fig3_abm_emergence.pdf
  fig3_abm_emergence.png
  fig4_abm_crossvalidation.pdf
  fig4_abm_crossvalidation.png
  fig5_mechanism_heatmap.pdf
  fig5_mechanism_heatmap.png
  fig6_hart_emergent_weights.pdf
  fig6_hart_emergent_weights.png
  fig7_complementarity.pdf
  fig7_complementarity.png
  fig8_complete_argument.pdf
  fig8_complete_argument.png
  figS1_correlated_activations.pdf
  figS1_correlated_activations